# 第 23 章 Bonus - 更現代的 OpenCV 物件追蹤器

In [ ]:
import cv2
import os
import numpy as np
import requests
from ultralytics import YOLO

## 下載 ViTTrack 模型

深度學習追蹤器需要模型檔，做法跟第 20、21 章下載 YuNet 與 SFace 的模型一樣。
我們從 OpenCV 官方的模型倉庫 OpenCV Zoo 取得它。

In [ ]:
os.makedirs("models", exist_ok=True)

VIT_PATH = "models/object_tracking_vittrack_2023sep.onnx"
VIT_URL = ("https://github.com/opencv/opencv_zoo/raw/main/models/"
           "object_tracking_vittrack/object_tracking_vittrack_2023sep.onnx")

if not os.path.exists(VIT_PATH):
    print("正在下載 ViTTrack 模型...")
    response = requests.get(VIT_URL)
    with open(VIT_PATH, "wb") as f:
        f.write(response.content)
    print("下載完成！")
else:
    print("模型檔案已存在，不需重複下載。")

print(f"模型大小：{os.path.getsize(VIT_PATH) / 1024:.0f} KB")


模型只有 700 KB 左右。作為對照，第 20 章那個 Haar Cascade 的 XML 是 930 KB。是相對輕量的深度學習模型。


## 使用現代 ViTTrack 追蹤器進行物件追蹤

下面直接沿用 23-E-1 節那套完整的追蹤系統（YOLO 週期性偵測、IoU 匹配維持 ID、軌跡繪製），
**只把建立追蹤器的部分換成 ViTTrack**，其餘邏輯都沒有改。

`init()` 與 `update()` 的用法跟 KCF 完全相同，所以整套系統可以直接替換。


In [ ]:
# --- IoU 計算函式（沿用 23-5-4） ---
def compute_iou(bbox1, bbox2):
    x1, y1 = bbox1[0], bbox1[1]
    x2, y2 = bbox1[0] + bbox1[2], bbox1[1] + bbox1[3]
    x3, y3 = bbox2[0], bbox2[1]
    x4, y4 = bbox2[0] + bbox2[2], bbox2[1] + bbox2[3]

    inter_x1, inter_y1 = max(x1, x3), max(y1, y3)
    inter_x2, inter_y2 = min(x2, x4), min(y2, y4)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    union_area = (x2-x1)*(y2-y1) + (x4-x3)*(y4-y3) - inter_area
    return inter_area / union_area if union_area > 0 else 0


In [ ]:
# --- 共用前處理函式（沿用 22-E） ---
def enhance_brightness(frame, alpha=1.2, beta=50):
    """調整影像的亮度與對比度，改善攝影機畫面偏暗的問題。"""
    return cv2.convertScaleAbs(frame, alpha=alpha, beta=beta)


In [ ]:
# --- 設定區 ---
TARGET_CLASS = "person"
CONF_THRESHOLD = 0.5
DETECT_INTERVAL = 30
IOU_THRESHOLD = 0.3
TRAIL_LENGTH = 300   # 軌跡保留的最大長度（幀數）
FRAME_SKIP = 2       # 每 N 幀只處理 1 幀（跳幀加速）

# --- 載入模型與影片 ---
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture("sample/video/walking.mp4")

trackers = []
tracker_ids = []
next_id = 1
frame_count = 0

trail_history = {}  # {tracker_id: [center1, center2, ...]}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # 跳幀處理：每 FRAME_SKIP 幀才處理一次，其餘幀直接跳過
    if frame_count % FRAME_SKIP != 0:
        continue

    # 前處理：調亮（視測試需求決定是否啟用）
    frame = enhance_brightness(frame)

    if frame_count % DETECT_INTERVAL == 0 or frame_count == FRAME_SKIP:
        results = model(frame, conf=CONF_THRESHOLD, verbose=False)
        new_bboxes = []

        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            if cls_name != TARGET_CLASS:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            new_bboxes.append((x1, y1, x2-x1, y2-y1))

        # 取得所有現有追蹤器的當前位置
        current_bboxes = []
        for tracker in trackers:
            success, bbox = tracker.update(frame)
            current_bboxes.append(bbox if success else None)

        # IoU 匹配
        matched_old = set()
        new_trackers = []
        new_ids = []

        for new_bbox in new_bboxes:
            best_iou, best_idx = 0, -1
            for i, cur_bbox in enumerate(current_bboxes):
                if cur_bbox is None or i in matched_old:
                    continue
                iou = compute_iou(new_bbox, cur_bbox)
                if iou > best_iou:
                    best_iou, best_idx = iou, i

            if best_iou >= IOU_THRESHOLD:
                trackers[best_idx].init(frame, new_bbox)
                new_trackers.append(trackers[best_idx])
                new_ids.append(tracker_ids[best_idx])
                matched_old.add(best_idx)
            else:
                # 與 23-E-1 的差異：把 KCF 換成 ViTTrack
                params = cv2.TrackerVit_Params()
                params.net = VIT_PATH
                tracker = cv2.TrackerVit_create(params)
                tracker.init(frame, new_bbox)
                new_trackers.append(tracker)
                new_ids.append(next_id)
                next_id += 1

        trackers = new_trackers
        tracker_ids = new_ids

    # 繪製階段：邊界框、編號、軌跡
    for i, tracker in enumerate(trackers):
        success, bbox = tracker.update(frame)
        if success:
            x, y, w, h = map(int, bbox)
            tid = tracker_ids[i]

            # 取追蹤器邊界框的上 1/3 位置作為軌跡點（接近人物的胸部位置）
            center = (x + w // 2, y + h // 3)
            if tid not in trail_history:
                trail_history[tid] = []
            trail_history[tid].append(center)

            # 限制軌跡長度
            if len(trail_history[tid]) > TRAIL_LENGTH:
                trail_history[tid].pop(0)

            # 繪製軌跡（紅線）
            if len(trail_history[tid]) > 1:
                pts = np.array(trail_history[tid], dtype=np.int32).reshape((-1, 1, 2))
                cv2.polylines(frame, [pts], isClosed=False,
                              color=(0, 0, 255), thickness=2,
                              lineType=cv2.LINE_AA)

            # 繪製邊界框與編號（綠色）
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2,
                          lineType=cv2.LINE_AA)
            cv2.putText(frame, f"{TARGET_CLASS} {tid}", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.imshow("YOLOv8 + ViTTrack | Trail Visualization", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
